# Imports

In [1]:
import os
import sys

from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib
import pickle
import re
import shutil
import yaml

from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
from itertools import product
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.pyplot as plt
from multiprocessing import Pool
import numpy as np
import pandas as pd
import seaborn as sns

from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log, load_yaml_fast

ROOT = str(Path.cwd().parent)

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Reproduction

## load data

In [11]:
root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/repro'
exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/o1-baselines'
exp_dirs += [entry.path for entry in os.scandir(root) if entry.is_dir()]

def process_exp_dir(exp_dir):
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        return None

    config = load_yaml_fast(os.path.join(setting_dir, 'config.yaml'))
    metric = load_yaml_fast(os.path.join(setting_dir, 'metrics.yaml'))
    result = config | metric
    result['exp_dir'] = exp_dir
    return result

# num_workers = os.cpu_count()
num_workers = 64

df = []
with ThreadPoolExecutor(max_workers=num_workers) as ex:
    futures = [ex.submit(process_exp_dir, exp_dir) for exp_dir in exp_dirs]

    for fut in as_completed(futures):
        r = fut.result()
        if r is not None:
            df.append(r)

df = pd.DataFrame(df)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)
df.loc[df['rec_lambda'] < 1, 'model'] = 'Time-o1'

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats_ext'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/repro.csv", index=False)

df.head(4)

,CSCM,ablation,activation,add_noise,affine,align_type,alpha,anomaly_ratio,auxi_batch_size,auxi_lambda,auxi_loss,auxi_mode,auxi_type,bandwidth,batch_size,begin_order,c_out,cf_depth,cf_dim,cf_drop,cf_head_dim,cf_heads,cf_mlp,cf_p,cf_weight_decay,channel_independence,checkpoints,cutoff_freq_percentage,d_ff,d_layers,d_model,data,data_id,data_path,data_percentage,dec_in,decomposition,des,devices,dilate_alpha,dist_scale,distance,distil,down_sampling_layers,down_sampling_method,down_sampling_window,dropout,e_layers,embed,enc_in,extra_metrics,factor,fc_dropout,features,first_order,fix_seed,freq,gamma,geomattn_dropout,gpu,head_dropout,hyper_dim,hyper_num,individual,inner_lr,inner_size,input_rank_ratio,input_reinit,input_trans,input_use_weights,inverse,is_training,itr,joint_forecast,k,kernel_size,l1_weight,label_len,learning_rate,leg_degree,load_from_disk,log_path,log_step,loss,lr_decay,lradj,m,mask_factor,mask_rate,max_iter,max_norm,meta_inner_steps,meta_lr,meta_optim_type,meta_type,min_lr,mlp_drop,mlp_hidden,mode,model,model_id,model_per_task,module_first,moving_avg,n_heads,noise_amp,noise_freq_percentage,noise_seed,noise_type,normalize,numItermax,num_freqs,num_kernels,num_tasks,num_workers,optim_type,ot_type,output_attention,output_log,output_pred,output_vis,overlap_ratio,p_hidden_dims,p_hidden_layers,padding_patch,patch_len,patch_num,patience,pca_dim,pct_start,pred_len,pretrain_model_path,rank_ratio,rec_lambda,reconstruction_type,reg_sk,reinit,report_to,requires_grad,rerun,results,revin,root_path,sample_attn,scale_factor,scales,seasonal_patterns,seq_len,shift,speedup_sklearn,step_size,stopThr,stride,subtract_last,target,task_name,test_batch_size,test_results,thread,top_k,train_epochs,use_amp,use_future_temporal_feature,use_gpu,use_multi_gpu,use_norm,use_nys,use_profiler,use_weights,verbose,warmup_steps,warping_length,weighting_type,window_size,wv,mae,mse,rmse,mape,mspe,mre,exp_dir,deterministic,chan_indep,extra_rev_in,input_trans_path,out_chan_indep
44,Bottleneck_Construct,0,gelu,False,0,0,0.5,0.25,1024,0.0,MAE,fft,complex,0.0,32,1,7,2,48,0.2,32,6,128,1,0,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,2048,1,512,ETTh1,ETTh1,ETTh1.csv,1.0,7,0,DLinear,"0,1,2,3",0.5,0.1,time,True,0,None,1,0.1,2,timeF,7,[],3,0.05,M,1,2023,h,0.01,0.5,0,0.0,2048,"[50, 20, 10]",0,0.0005,5,1.0,0,None,0,False,1,1,0,3,24,0.00005,48,0.0005,2,,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,10,MSE,0.5,type1,3,0.01,0.25,10,1.0,1,0.0005,sgd,all,0.000001,0.3,64,0,DLinear,ETTh1_96_96,0,1,25,8,1,0.05,2023,sin,1,10000,16,6,5,10,adam,emd1d_h,False,False,False,False,0.15,"[128, 128]",2,end,16,14,3,all,0.2,96,None,1.0,1.0,imputation,0.1,0,local,True,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT...,0,2,"[16, 8, 4, 2, 1]",Monthly,96,0,0,1,0.0001,8,0,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,0,True,False,True,0,0,0,0,20,96,softmax,"[4, 4]",db1,0.403672,0.388942,0.623652,8.601435,37319.710938,8.601435,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.0,0.0,0.0,None,1.0
31,Bottleneck_Construct,0,gelu,False,0,0,0.5,0.25,1024,0.0,MAE,fft,complex,0.0,32,1,7,2,48,0.2,32,6,128,1,0,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,2048,1,512,ETTh1,ETTh1,ETTh1.csv,1.0,7,0,DLinear,"0,1,2,3",0.5,0.1,time,True,0,None,1,0.1,2,timeF,7,[],3,0.05,M,1,2023,h,0.01,0.5,0,0.0,2048,"[50, 20, 10]",0,0.0005,5,1.0,0,None,0,False,1,1,0,3,24,0.00005,48,0.0005,2,,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,10,MSE,0.5,type1,3,0.01,0.25,10,1.0,1,0.0005,sgd,all,0.000001,0.3,64,0,DLinear,ETTh1_96_192,0,1,25,8,1,0.05,2023,sin,1,10000,16,6,5,10,adam,emd1d_h,False,False,False,False,0.15,"[128, 128]",2,end,16,14,3,all,0.2,192,None,1.0,1.0,imputation,0.1,0,local,True,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/ETT...,0,2,"[16, 8, 4, 2, 1]",Monthly,96,0,0,1,0.0001,8,0,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_

## analysis

In [13]:
min_mode = 'each'

df2 = df.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'align_type', 'individual']
if min_mode == 'group':
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    df2['sum_error'] = df2['mse'] + df2['mae']
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'inner_lr', 'meta_lr', 'auxi_loss', 'rec_lambda', 'auxi_lambda', 'rank_ratio', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'first_order', 'auxi_mode', 'auxi_type', 'pca_dim', 'reinit', 'use_weights', 'sample_attn', 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Time-o1', 'DLinear']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

# save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats'
# os.makedirs(save_root, exist_ok=True)
# df2.to_csv(f"{save_root}/finetune_best_meta.csv", index=False)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

# pl_order = [96, 192, 336, 720, 'Avg']
# df2['pred_len'] = pd.Categorical(df2['pred_len'], categories=pl_order, ordered=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

df2_show = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in df2_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
df2_show = df2_show[columns]
df2_show


/tmp/ipykernel_2438200/2182416813.py:36: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model              Time-o1             DLinear          
                       mse       mae       mse       mae
data_id pred_len                                        
ETTm1   96        0.321138  0.357436  0.345850  0.373350
        192       0.359580  0.378119  0.380310  0.389934
        336       0.389238  0.399987  0.413243  0.413936
        720       0.447020  0.434891  0.471745  0.449863
        Avg       0.379244  0.392608  0.402787  0.406771
ETTm2   96        0.172235  0.251189  0.187883  0.282945
        192       0.235445  0.293936  0.279695  0.355772
        336       0.293289  0.332727  0.375375  0.420250
        720       0.387706  0.389056  0.525772  0.508116
        Avg       0.272169  0.316727  0.342181  0.391771
ETTh1   96        0.368002  0.390765  0.388942  0.403672
        192       0.424089  0.421955  0.442218  0.439876
        336       0.466960  0.441356  0.488474  0.466596
        720       0.464932  0.463144  0.505058  0.502427
        Avg       0.430996  0.429305  0.456173  0.453143
ETTh2   96        0.281609  0.330220  0.330024  0.382585
        192       0.359048  0.380537  0.439167  0.449528
        336       0.393813  0.413791  0.589380  0.537515
        720       0.400099  0.426993  0.757178  0.626219
        Avg       0.358642  0.387885  0.528937  0.498962
ECL     96        0.144906  0.234782       NaN       NaN
        192       0.158948  0.248666       NaN       NaN
        336       0.173076  0.264487       NaN       NaN
        720       0.203276  0.292044       NaN       NaN
        Avg       0.170051  0.259995       NaN       NaN
Traffic 96        0.392572  0.264985       NaN       NaN
        192       0.409847  0.274966       NaN       NaN
        336       0.420893  0.280186       NaN       NaN
        720       0.450731  0.298500       NaN       NaN
        Avg       0.418511  0.279659       NaN       NaN
Weather 96        0.169709  0.219077       NaN       NaN
        192       0.211001  0.261077       NaN       NaN
        336       0.258361  0.296796       NaN       NaN
        720       0.327872  0.350033       NaN       NaN
        Avg       0.241736  0.281746       NaN       NaN
PEMS03  12        0.069636  0.176462       NaN       NaN
        24        0.087209  0.198611       NaN       NaN
        36        0.105917  0.219741       NaN       NaN
        48        0.125593  0.240792       NaN       NaN
        Avg       0.097089  0.208902       NaN       NaN
PEMS08  12        0.081086  0.183092       NaN       NaN
        24        0.117261  0.217823       NaN       NaN
        36        0.156879  0.252824       NaN       NaN
        48        0.207103  0.294117       NaN       NaN
        Avg       0.140582  0.236964       NaN       NaN

# Time-o1 Extension

## load data

In [2]:
# root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/long_term_ext'
# exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

# root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/long_term_channel'
# exp_dirs += [entry.path for entry in os.scandir(root) if entry.is_dir()]

# root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/long_term_Tucker'
# exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]

root = '/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/Time-o1/results/long_term_KronPCA'
exp_dirs = [entry.path for entry in os.scandir(root) if entry.is_dir()]


def process_exp_dir(exp_dir):
    runned, setting_dir = exist_metric(exp_dir)
    if not runned or not os.path.exists(os.path.join(setting_dir, 'config.yaml')):
        return None

    config = load_yaml_fast(os.path.join(setting_dir, 'config.yaml'))
    metric = load_yaml_fast(os.path.join(setting_dir, 'metrics.yaml'))
    result = config | metric
    result['exp_dir'] = exp_dir
    return result

# num_workers = os.cpu_count()
num_workers = 64

df = []
with ThreadPoolExecutor(max_workers=num_workers) as ex:
    futures = [ex.submit(process_exp_dir, exp_dir) for exp_dir in exp_dirs]

    for fut in as_completed(futures):
        r = fut.result()
        if r is not None:
            df.append(r)

df = pd.DataFrame(df)
df['input_pca_dim'] = df['input_pca_dim'].fillna(df['pca_dim'])

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats_ext'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/long_term_ext.csv", index=False)

df.head(4)

,CSCM,ablation,activation,add_noise,affine,align_type,alpha,anomaly_ratio,auxi_batch_size,auxi_lambda,auxi_loss,auxi_mode,auxi_type,bandwidth,batch_size,begin_order,c_out,cf_depth,cf_dim,cf_drop,cf_head_dim,cf_heads,cf_mlp,cf_p,cf_weight_decay,chan_indep,channel_independence,checkpoints,cutoff_freq_percentage,d_ff,d_layers,d_model,data,data_id,data_path,data_percentage,dec_in,decomposition,des,deterministic,devices,dilate_alpha,dist_scale,distance,distil,down_sampling_layers,down_sampling_method,down_sampling_window,dropout,e_layers,embed,enc_in,extra_metrics,extra_rev_in,factor,fc_dropout,features,first_order,fix_seed,freq,gamma,geomattn_dropout,gpu,head_dropout,hyper_dim,hyper_num,individual,inner_lr,inner_size,input_pca_dim,input_rank_ratio,input_reinit,input_trans,input_trans_path,input_use_weights,inverse,is_training,itr,joint_forecast,k,kernel_size,l1_weight,label_len,learning_rate,leg_degree,load_from_disk,log_path,log_step,loss,lr_decay,lradj,m,mask_factor,mask_rate,max_iter,max_norm,meta_inner_steps,meta_lr,meta_optim_type,meta_type,min_lr,mlp_drop,mlp_hidden,mode,model,model_id,model_per_task,module_first,moving_avg,n_heads,noise_amp,noise_freq_percentage,noise_seed,noise_type,normalize,numItermax,num_freqs,num_kernels,num_tasks,num_workers,optim_type,ot_type,out_chan_indep,output_attention,output_log,output_pred,output_vis,overlap_ratio,p_hidden_dims,p_hidden_layers,padding_patch,patch_len,patch_num,patience,pca_dim,pca_iter_max,pca_tol,pct_start,pred_len,pretrain_model_path,rank_ratio,rec_lambda,reconstruction_type,reg_sk,reinit,report_to,requires_grad,rerun,results,revin,root_path,sample_attn,scale_factor,scales,seasonal_patterns,seq_len,shift,speedup_sklearn,step_size,stopThr,stride,subtract_last,target,task_name,test_batch_size,test_results,thread,top_k,train_epochs,use_amp,use_future_temporal_feature,use_gpu,use_multi_gpu,use_norm,use_nys,use_profiler,use_weights,verbose,warmup_steps,warping_length,weighting_type,window_size,wv,mae,mse,rmse,mape,mspe,mre,exp_dir
3020,Bottleneck_Construct,0,gelu,False,0,0,0.5,0.25,1024,0.9,MAE,basis,pca,0.0,32,1,21,2,48,0.2,32,6,128,1,0,0,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,2048,1,512,custom_PCA,Weather,weather.csv,1.0,21,0,FreTS,0,"0,1,2,3",0.5,0.1,time,True,0,None,1,0.1,2,timeF,21,[],0,3,0.05,M,1,2023,h,0.01,0.5,0,0.0,2048,"[50, 20, 10]",0,0.0005,5,T,1.0,0,None,None,0,False,1,1,0,3,24,0.00005,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,10,MSE,0.5,type1,3,0.01,0.25,10,1.0,1,0.0005,sgd,all,0.000001,0.3,64,0,FreTS,Weather_96_96,0,1,25,8,1,0.05,2023,sin,1,10000,16,6,5,10,adam,emd1d_h,1,False,False,False,False,0.15,"[128, 128]",2,end,16,14,10,KronPCA,500,0.000001,0.2,96,None,"[0.1, 0.9]",0.1,imputation,0.1,1,local,True,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,/mnt/tidalfs-bdsz01/usr/panlicheng/dataset/wea...,0,2,"[16, 8, 4, 2, 1]",Monthly,96,0,2,1,0.0001,8,0,OT,long_term_forecast,1,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,1,5,10,False,0,True,False,True,0,0,0,0,20,96,softmax,"[4, 4]",db1,0.235278,0.178475,0.422463,2.945541,66851.882812,2.945541,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
3022,Bottleneck_Construct,0,gelu,False,0,0,0.5,0.25,1024,0.9,MAE,basis,pca,0.0,32,1,21,2,48,0.2,32,6,128,1,0,0,0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,0.06,2048,1,512,custom_PCA,Weather,weather.csv,1.0,21,0,FreTS,0,"0,1,2,3",0.5,0.1,time,True,0,None,1,0.1,2,timeF,21,[],0,3,0.05,M,1,2023,h,0.01,0.5,0,0.0,2048,"[50, 20, 10]",0,0.0005,5,T,1.0,0,None,None,0,False,1,1,0,3,24,0.00005,48,0.001,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...,10,MSE,0.5,type1,3,0.01,0.25,10,1.0,1,0.0005,sgd,all,0.000001,0.3,64,0,FreTS,Weather_96_96,0,1,25,8,1,0.05,2023,sin,1,10000,16,6,5,10,adam,emd1d_h,1,False,False,False,False,0.15,"[128, 128]",2,end,16,14,10,KronPCA,500,0.000001,0.2,96,None,"[0.1, 0.7]",0.1,imputation,0.1,1,local,True,0,/mnt/t

## pre-load

In [ ]:
save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats_ext'
df = pd.read_csv(f'{save_root}/long_term_ext.csv')

df.head(4)

## query

In [6]:

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'batch_size', "lradj", "train_epochs", "patience", "auxi_mode", "auxi_type", "pca_dim", "rank_ratio", "reinit", "use_weights", "auxi_loss", 'pca_iter_max', 'pca_tol', "out_chan_indep", "speedup_sklearn", 'exp_dir']

df.query('model == "Fredformer" and data_id == "ETTh1" and pred_len == 336').sort_values(by=['mae']).head(10)[columns]
df.query('model == "Fredformer" and data_id == "ETTh2" and pred_len == 96').sort_values(by=['mae']).head(10)[columns]
df.query('model == "Fredformer" and data_id == "ETTm1" and pred_len == 192').sort_values(by=['mae']).head(10)[columns]
df.query('model == "Fredformer" and data_id == "ETTm1" and pred_len == 720').sort_values(by=['mse']).head(10)[columns]
# df.query('model == "Fredformer" and data_id == "ETTm2" and pred_len == 192').sort_values(by=['mse']).head(10)[columns]
# df.query('model == "FreTS" and data_id == "Weather" and pred_len == 336').sort_values(by=['mae']).head(10)[columns]
df.query('model == "iTransformer" and data_id == "Traffic" and pred_len == 336').sort_values(by=['mse']).head(10)[columns]
df.query('model == "iTransformer" and data_id == "Traffic" and pred_len == 720').sort_values(by=['mse']).head(10)[columns]

,model,pred_len,data_id,mse,mae,learning_rate,rec_lambda,auxi_lambda,batch_size,lradj,train_epochs,patience,auxi_mode,auxi_type,pca_dim,rank_ratio,reinit,use_weights,auxi_loss,pca_iter_max,pca_tol,out_chan_indep,speedup_sklearn,exp_dir
19170,iTransformer,720,Traffic,0.451968,0.298180,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.8, 0.9]",0,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19202,iTransformer,720,Traffic,0.452132,0.298678,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[1.0, 0.9]",0,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19227,iTransformer,720,Traffic,0.452264,0.298782,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.8, 0.8]",1,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19214,iTransformer,720,Traffic,0.452382,0.298645,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.8, 0.8]",0,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19186,iTransformer,720,Traffic,0.452515,0.298995,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.6, 0.9]",1,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19196,iTransformer,720,Traffic,0.452741,0.299401,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[1.0, 0.7]",1,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19201,iTransformer,720,Traffic,0.452961,0.299681,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[1.0, 0.9]",1,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19204,iTransformer,720,Traffic,0.453406,0.299387,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.8, 0.7]",0,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19226,iTransformer,720,Traffic,0.453579,0.299401,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[0.6, 0.8]",1,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
19233,iTransformer,720,Traffic,0.453722,0.299428,0.001,0.8,0.2,8,type1,10,3,basis,pca,KronPCA,"[1.0, 0.8]",0,0,MAE,500,0.000001,1,2,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...


## selection

In [4]:
dfs = df.copy()

# 每个 condition 就是一个 dict，平铺写，跟原来差不多
conditions = [
    # ETTh1
    dict(model="Fredformer", data_id="ETTh1", pred_len=336, auxi_type="pca", auxi_mode="basis", pca_dim="KronPCA", rank_ratio=[1.0, 0.8], reinit=1, use_weights=0, pca_iter_max=500, pca_tol=1e-6, learning_rate=0.001, rec_lambda=0.0, batch_size=128, lradj="type1", auxi_loss="MAE", out_chan_indep=1),
    # ETTh2
    # dict(model="Fredformer", data_id="ETTh1", pred_len=96, auxi_type="pca", auxi_mode="basis", pca_dim="KronPCA", rank_ratio=[1.0, 0.9], reinit=0, use_weights=0, pca_iter_max=500, pca_tol=1e-6, learning_rate=0.001, rec_lambda=0.0, batch_size=128, lradj="type1", auxi_loss="MAE", out_chan_indep=1),
]
cond_df = pd.DataFrame(conditions)
def _to_hashable(x): return tuple(x) if isinstance(x, list) else x
GROUP_KEYS = ['model', 'data_id', 'pred_len']
all_cols = cond_df.columns.tolist()

df_h = dfs.copy()
df_h[all_cols] = df_h[all_cols].applymap(_to_hashable)   # 新版 pandas 用 .map
cond_h = cond_df.applymap(_to_hashable)

# 1) 标记每行属于的 group 是否在 conditions 中
group_has_cond = df_h[GROUP_KEYS].merge(
    cond_h[GROUP_KEYS].drop_duplicates(),
    how='left', on=GROUP_KEYS, indicator=True
)['_merge'] == 'both'

# 2) 标记每行是否完全匹配某个 condition（所有列都相等）
row_matches_cond = df_h[all_cols].merge(
    cond_h, how='left', on=all_cols, indicator=True
)['_merge'] == 'both'

# 3) 最终保留：不在 condition group 的行  OR  在 group 中且匹配 condition 的行
mask = (~group_has_cond) | row_matches_cond
dfs = dfs[mask.values].reset_index(drop=True)

/tmp/ipykernel_420290/2180895703.py:16: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_h[all_cols] = df_h[all_cols].applymap(_to_hashable)   # 新版 pandas 用 .map
/tmp/ipykernel_420290/2180895703.py:17: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  cond_h = cond_df.applymap(_to_hashable)


## analysis

In [5]:
min_mode = 'sum'

df2 = dfs.copy()

if min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    # min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len', 'input_pca_dim'])['mae'].idxmin()
    df2 = df2.loc[min_mse_idx]
elif min_mode == 'sum':
    # 新增：mse+mae最小
    # w_mse = 0.0
    # w_mse = 0.1
    # w_mse = 0.2
    # w_mse = 0.3
    w_mse = 0.5
    # w_mse = 0.9
    # w_mse = 0.8
    # w_mse = 1.0

    df2['sum_error'] = df2['mse'] * w_mse + df2['mae'] * (1 - w_mse)
    min_sum_idx = df2.groupby(['model', 'data_id', 'pred_len'])['sum_error'].idxmin()
    # min_sum_idx = df2.groupby(['model', 'data_id', 'input_pca_dim', 'pred_len'])['sum_error'].idxmin()
    df2 = df2.loc[min_sum_idx]
    df2 = df2.drop(columns=['sum_error'])

# columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'batch_size', "lradj", "train_epochs", "patience", "auxi_mode", "auxi_type", "input_pca_dim", "pca_dim", "rank_ratio", "reinit", "use_weights", "auxi_loss", "input_trans", "input_use_weights", "input_reinit", "input_rank_ratio", "chan_indep", "extra_rev_in", "out_chan_indep", "speedup_sklearn", 'exp_dir']
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'batch_size', "lradj", "train_epochs", "patience", "auxi_mode", "auxi_type", "pca_dim", "rank_ratio", "reinit", "use_weights", "auxi_loss", 'pca_iter_max', 'pca_tol', "out_chan_indep", "speedup_sklearn", 'exp_dir']
df2 = df2[columns]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
# dst_order = ['ETTh1']
df2 = df2.query('data_id in @dst_order')
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'iTransformer', 'FreTS', 'MICN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)
df2.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)
# df2.sort_values(by=['model', 'data_id', 'input_pca_dim', 'pred_len'], inplace=True)

# save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/TransDF/stats_ext'
# os.makedirs(save_root, exist_ok=True)
# df2.to_csv(f"{save_root}/long_term_ext_best.csv", index=False)

# print(df2)
df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
# df2_avg = df2.groupby(['model', 'data_id', 'input_pca_dim']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)

# pl_order = [96, 192, 336, 720, 'Avg']
# df2['pred_len'] = pd.Categorical(df2['pred_len'], categories=pl_order, ordered=True)

df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
# df2.sort_values(by=['data_id', 'model', 'input_pca_dim', 'pred_len'], inplace=True)
df2.dropna(inplace=True, thresh=5)
df2

# df2_show = df2.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
# columns = []
# for model in df2_show.columns.levels[0]:
#     columns.append((model, 'mse'))
#     columns.append((model, 'mae'))
# df2_show = df2_show[columns]
# df2_show


/tmp/ipykernel_420290/1455876942.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,learning_rate,rec_lambda,auxi_lambda,batch_size,lradj,train_epochs,patience,auxi_mode,auxi_type,pca_dim,rank_ratio,reinit,use_weights,auxi_loss,pca_iter_max,pca_tol,out_chan_indep,speedup_sklearn,exp_dir
0,Fredformer,96,ETTm1,0.313106,0.351796,0.001000,0.0000,1.0000,128.0,type1,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",1.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
1,Fredformer,192,ETTm1,0.351947,0.374000,0.002000,0.0000,1.0000,512.0,type1,100.0,10.0,basis,pca,KronPCA,"[1.0, 0.8]",0.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
2,Fredformer,336,ETTm1,0.382921,0.395507,0.002000,0.0000,1.0000,512.0,type1,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",1.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
3,Fredformer,720,ETTm1,0.445766,0.432135,0.002000,0.0000,1.0000,256.0,type1,100.0,10.0,basis,pca,KronPCA,"[1.0, 0.8]",1.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
36,Fredformer,Avg,ETTm1,0.373435,0.388360,0.001750,0.0000,1.0000,352.0,NaN,100.0,10.0,NaN,NaN,NaN,NaN,0.75,0.0,NaN,500.0,0.000001,1.0,2.0,NaN
4,Fredformer,96,ETTm2,0.168998,0.249519,0.001000,0.0000,1.0000,256.0,type1,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",1.00,0.0,MAE,500.0,0.000001,0.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
5,Fredformer,192,ETTm2,0.234687,0.293234,0.000200,0.0000,1.0000,128.0,TST,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",1.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
6,Fredformer,336,ETTm2,0.291360,0.330715,0.002000,0.0000,1.0000,512.0,type3,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",0.00,0.0,MAE,500.0,0.000001,1.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
7,Fredformer,720,ETTm2,0.386195,0.386792,0.002000,0.0000,1.0000,128.0,type3,100.0,10.0,basis,pca,KronPCA,"[1.0, 1.0]",0.00,0.0,MAE,500.0,0.000001,0.0,2.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
37,Fredformer,Avg,ETTm2,0.270310,0.315065,0.001300,0.0000,1.0000,256.0,NaN,100.0,10.0,NaN,NaN,NaN,NaN,0.50,0.0,NaN,500.0,0.000001,0.5,2.0,NaN


In [8]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'rank_ratio', 'batch_size', 'auxi_mode', 'auxi_type', 'module_first', 'auxi_loss', 'use_weights', 'reinit', 'pca_dim', 'input_reinit', 'input_rank_ratio', 'input_use_weights', 'input_trans', 'chan_indep', 'extra_rev_in', 'out_chan_indep', 'lradj', 'train_epochs', 'patience', 'exp_dir']
df2.dropna(thresh=5)[columns].query('model == "Fredformer" and data_id == "ETTh1"')
df2.dropna(thresh=5)[columns].query('model == "Fredformer" and data_id == "ETTh2"')
df2.dropna(thresh=5)[columns].query('model == "Fredformer" and data_id == "ETTm1"')
df2.dropna(thresh=5)[columns].query('model == "Fredformer" and data_id == "ETTm2"')

,model,pred_len,data_id,mse,mae,learning_rate,rec_lambda,rank_ratio,batch_size,auxi_mode,auxi_type,module_first,auxi_loss,use_weights,reinit,pca_dim,input_reinit,input_rank_ratio,input_use_weights,input_trans,chan_indep,extra_rev_in,out_chan_indep,lradj,train_epochs,patience,exp_dir
4,Fredformer,96,ETTm2,0.171492,0.251140,0.00200,0.0,1.0,128.0,basis,pca,1.0,MAE,0.0,1.0,T,0.0,1.0,0.0,evd,0.0,1.0,1.0,TST,100.0,10.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
5,Fredformer,192,ETTm2,0.236244,0.294360,0.00200,0.0,0.9,128.0,basis,pca,1.0,MAE,0.0,1.0,T,0.0,1.0,0.0,evd,0.0,1.0,1.0,TST,100.0,10.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
6,Fredformer,336,ETTm2,0.296143,0.331933,0.00200,0.0,0.8,128.0,basis,pca,1.0,MAE,0.0,1.0,T,0.0,1.0,0.0,evd,0.0,1.0,1.0,TST,100.0,10.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
7,Fredformer,720,ETTm2,0.393065,0.389886,0.00020,0.0,0.9,128.0,basis,pca,1.0,MAE,0.0,1.0,T,0.0,1.0,0.0,evd,0.0,1.0,1.0,TST,100.0,10.0,/mnt/tidalfs-bdsz01/dataset/llm_ckpt/plc_data/...
17,Fredformer,Avg,ETTm2,0.274236,0.316830,0.00155,0.0,0.9,128.0,NaN,NaN,1.0,NaN,0.0,1.0,NaN,0.0,1.0,0.0,NaN,0.0,1.0,1.0,NaN,100.0,10.0,NaN


# report to latex

## load and merge data

In [10]:
save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
dfb = pd.read_csv(f"{save_root}/baselines_chosen.csv")

min_mode = 'each'
columns = ['model', 'data_id', 'learning_rate', 'batch_size', 'patience', 'individual', 'train_epochs']
if min_mode == 'group':
    mse_mean = dfb.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    dfb = dfb.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = dfb.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    dfb = dfb.loc[min_mse_idx]

dfb = dfb[['model', 'pred_len', 'data_id', 'mse', 'mae']]
datasets = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
dfb = dfb[dfb['data_id'].isin(datasets)]


save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
dff = pd.read_csv(f'{save_root}/finetune_best.csv')

dff = dff[
    ((dff.data_id == 'ETTm1') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTm2') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTh1') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ETTh2') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'ECL') & (dff.model == 'TQNet')) |
    # ((dff.data_id == 'Traffic') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'Weather') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'PEMS03') & (dff.model == 'TQNet')) |
    ((dff.data_id == 'PEMS08') & (dff.model == 'TQNet'))
]

dff['model'] = 'QDF'
dff.to_csv(f"{save_root}/long_term.csv", index=False)
dff = dff[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dft = pd.concat([dfb, dff], axis=0)

# dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
dft['data_id'] = pd.Categorical(dft['data_id'], categories=dst_order, ordered=True)

model_order = ['QDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
dft = dft[dft['model'].isin(model_order)]
dft['model'] = pd.Categorical(dft['model'], categories=model_order, ordered=True)

dft_avg = dft.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
dft_avg['pred_len'] = 'Avg'

dft = pd.concat([dft, dft_avg]).reset_index(drop=True)
dft.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

dft_show = dft.set_index(['data_id', 'pred_len', 'model']).unstack('model').swaplevel(axis=1)
columns = []
for model in dft_show.columns.levels[0]:
    columns.append((model, 'mse'))
    columns.append((model, 'mae'))
dft_show = dft_show[columns]
dft_show

/tmp/ipykernel_1486806/2195932971.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dft_avg = dft.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


model                  QDF               TQNet                 PDF            \
                       mse       mae       mse       mae       mse       mae   
data_id pred_len                                                               
ETTm1   96        0.306602  0.348971  0.310382  0.351721  0.326234  0.363356   
        192       0.352414  0.376268  0.356100  0.377443  0.364817  0.381136   
        336       0.382595  0.397512  0.387569  0.399900  0.396639  0.402112   
        720       0.441163  0.434478  0.450021  0.436546  0.458370  0.437322   
        Avg       0.370693  0.389307  0.376018  0.391403  0.386515  0.395981   
ETTm2   96        0.170291  0.252600  0.174746  0.255768  0.176168  0.263647   
        192       0.233855  0.293836  0.242704  0.300489  0.245158  0.310199   
        336       0.290130  0.331435  0.297452  0.336435  0.305491  0.345152   
        720       0.386822  0.388532  0.394070  0.393085  0.403959  0.403241   
        Avg       0.270274  0.316601  0.277243  0.321444  0.282694  0.330560   
ETTh1   96        0.365111  0.388897  0.372007  0.391440  0.388068  0.400122   
        192       0.427475  0.421499  0.430498  0.424173  0.440350  0.428146   
        336       0.465918  0.448515  0.486262  0.454420  0.483253  0.448762   
        720       0.466485  0.466534  0.506832  0.485931  0.495008  0.481555   
        Avg       0.431247  0.431361  0.448900  0.438991  0.451670  0.439646   
ETTh2   96        0.285880  0.337922  0.293373  0.342665  0.290784  0.340032   
        192       0.361035  0.388219  0.363695  0.390391  0.374348  0.390743   
        336       0.407637  0.422494  0.411333  0.424371  0.413972  0.425852   
        720       0.419220  0.438823  0.429976  0.443821  0.421089  0.439893   
        Avg       0.368443  0.396864  0.374594  0.400312  0.375048  0.399130   
ECL     96        0.134708  0.228838  0.143456  0.237354  0.175417  0.259347   
        192       0.153000  0.244973  0.160641  0.252081  0.181722  0.265927   
        336       0.169068  0.262404  0.178238  0.269703  0.197201  0.281979   
        720       0.201599  0.290306  0.217629  0.302696  0.237404  0.315331   
        Avg       0.164594  0.256630  0.174991  0.265458  0.197936  0.280646   
Weather 96        0.158300  0.200693  0.159965  0.202723  0.181140  0.221251   
        192       0.206637  0.244936  0.209872  0.247166  0.231525  0.262294   
        336       0.262811  0.286290  0.266696  0.288953  0.285225  0.299993   
        720       0.342250  0.339154  0.346121  0.342470  0.360232  0.348378   
        Avg       0.242499  0.267768  0.245663  0.270328  0.264530  0.282979   
PEMS03  12        0.064309  0.167092  0.097305  0.179638  0.091645  0.204271   
        24        0.080374  0.188680  0.098929  0.204429  0.148696  0.260927   
        36        0.097987  0.208369  0.123181  0.229999  0.210095  0.313705   
        48        0.111981  0.223476  0.157448  0.255610  0.274832  0.364260   
        Avg       0.088663  0.196904  0.119216  0.217419  0.181317  0.285790   
PEMS08  12        0.074487  0.175849  0.078706  0.182539  0.099847  0.208661   
        24        0.104101  0.207774  0.117204  0.222451  0.167697  0.272787   
        36        0.134275  0.236954  0.157844  0.260074  0.244455  0.333303   
        48        0.168359  0.263294  0.203122  0.295390  0.327415  0.388973   
        Avg       0.120305  0.220968  0.139219  0.240114  0.209853  0.300931   

model            Fredformer           iTransformer               FreTS  \
                        mse       mae          mse       mae       mse   
data_id pred_len                                                         
ETTm1   96         0.326369  0.360869     0.337877  0.372283  0.341839   
        192        0.365194  0.382132     0.381666  0.396060  0.384536   
        336        0.395987  0.404369     0.426993  0.423962  0.415986   
        720        0.459217  0.444342     0.495750  0.462558  0.513302   
        Avg        0.386692  0.397928    

## save to latex table

In [11]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



# dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['QDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if i == 0:
            title = r'\multirow{5}{*}{{\rotatebox{90}{\scalebox{0.95}{' + dst + '}}}}'
            contents.append(title)
        line = f"& {pl} "
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
        if i == 3:
            contents.append(r"\cmidrule(lr){2-24}")
        elif i  == 4:
            contents.append("\\midrule\n")

stats_title = r"\multicolumn{2}{c|}{\scalea{{$1^{\text{st}}$ Count}}}"  # 标题：合并前两列
stats_values = []
for model in model_order:
    mse = mse_count[model]
    mae = mae_count[model]
    # 次数>0时用\bst{}标记，否则0
    mse_str = r"\scalea{\bst{" + str(mse) + "}}" if mse > 0 else r"\scalea{" + str(mse) + "}"
    mae_str = r"\scalea{\bst{" + str(mae) + "}}" if mae > 0 else r"\scalea{" + str(mae) + "}"
    stats_values.append(f"{mse_str} & {mae_str}")  # 每个模型的MSE/MAE次数
# 合并统计行
stats_line = f"{stats_title} & {' & '.join(stats_values)} \\\\"
stats_contents.append(stats_line)

print(sum(mse_count.values()), sum(mae_count.values()))
print(max(mse_count.values()), max(mae_count.values()))

with open(f"{save_root}/long_term.tex", "w") as f:
    f.write("\n".join(contents + stats_contents))

40 40
39 39


/tmp/ipykernel_1486806/975220537.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_1486806/975220537.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_1486806/975220537.py:35: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

## write avg

In [13]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['MetaDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if pl != 'Avg':
            continue
        line = r"\multicolumn{2}{l}{\scalea{" + dst + r"}}" + '\n'
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
    if dst != dst_order[-1]:
        contents.append("\\midrule")


with open(f"{save_root}/long_term_avg.tex", "w") as f:
    f.write("\n".join(contents))
print("\n".join(contents))

\multicolumn{2}{l}{\scalea{ETTm1}}
& \scalea{\bst{0.371}} & \scalea{\bst{0.389}}& \scalea{\subbst{0.376}} & \scalea{\subbst{0.391}}& \scalea{0.387} & \scalea{0.396}& \scalea{0.387} & \scalea{0.398}& \scalea{0.411} & \scalea{0.414}& \scalea{0.414} & \scalea{0.421}& \scalea{0.438} & \scalea{0.430}& \scalea{0.396} & \scalea{0.421}& \scalea{0.413} & \scalea{0.407}& \scalea{0.389} & \scalea{0.400}& \scalea{0.403} & \scalea{0.407} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTm2}}
& \scalea{\bst{0.270}} & \scalea{\bst{0.317}}& \scalea{\subbst{0.277}} & \scalea{\subbst{0.321}}& \scalea{0.283} & \scalea{0.331}& \scalea{0.280} & \scalea{0.324}& \scalea{0.295} & \scalea{0.336}& \scalea{0.316} & \scalea{0.365}& \scalea{0.302} & \scalea{0.334}& \scalea{0.308} & \scalea{0.364}& \scalea{0.286} & \scalea{0.328}& \scalea{0.303} & \scalea{0.344}& \scalea{0.342} & \scalea{0.392} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTh1}}
& \scalea{\bst{0.431}} & \scalea{\bst{0.431}}& \scalea{0.449} & \scalea{0.439}& \sc

/tmp/ipykernel_253782/3944324592.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index

## write avg

In [22]:
save_root = "/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3"
# dft_show.to_latex(f"{save_root}/long_term.tex")



dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather', 'PEMS03', 'PEMS08']
model_order = ['MetaDF', 'TQNet', 'PDF', 'Fredformer', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'PatchTST', 'DLinear']
# 初始化统计字典：每个模型的MSE第1次数和MAE第1次数
mse_count = {model: 0 for model in model_order}
mae_count = {model: 0 for model in model_order}


contents = []
stats_contents = []  # 存储统计行内容
for dst in dst_order:
    dfi = dft[dft.data_id == dst]

    dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
    dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')

    # MSE第1名：mse_rank=1的模型
    mse_first_models = dfi[dfi['mse_rank'] == 1]['model']
    for model in mse_first_models:
        if model in mse_count:  # 确保模型在model_order中
            mse_count[model] += 1
    
    # MAE第1名：mae_rank=1的模型
    mae_first_models = dfi[dfi['mae_rank'] == 1]['model']
    for model in mae_first_models:
        if model in mae_count:
            mae_count[model] += 1

    for metric in ['mse', 'mae']:
        dfi[f'{metric}_3f'] = dfi[metric].apply(lambda x: "{:.3f}".format(x))
        # 按排名标记：第1名→\bst{}，第2名→\subbst{}，其他→原格式
        conditions = [dfi[f'{metric}_rank'] == 1, dfi[f'{metric}_rank'] == 2]
        choices = [r'\bst{' + dfi[f'{metric}_3f'] + '}', r'\subbst{' + dfi[f'{metric}_3f'] + '}']
        dfi[f'{metric}_processed'] = np.select(conditions, choices, default=dfi[f'{metric}_3f'])
    dfi = dfi[['model', 'pred_len', 'mse_processed', 'mae_processed']]
    dfi.columns = ['model', 'pred_len', 'mse', 'mae']
    dfi['mse'] = dfi['mse'].apply(lambda x: r"\scalea{" + x + "}")  # 应用\scalea
    dfi['mae'] = dfi['mae'].apply(lambda x: r"\scalea{" + x + "}")
    for i, (pl, group) in enumerate(dfi.groupby('pred_len')):
        if pl != 'Avg':
            continue
        line = r"\multicolumn{2}{l}{\scalea{" + dst + r"}}" + '\n'
        for j, row in enumerate(group.itertuples(index=False)):
            line += f"& {row.mse} & {row.mae}"
        line += " \\\\"
        contents.append(line)
    if dst != dst_order[-1]:
        contents.append("\\midrule")


with open(f"{save_root}/long_term_avg.tex", "w") as f:
    f.write("\n".join(contents))
print("\n".join(contents))

\multicolumn{2}{l}{\scalea{ETTm1}}
& \scalea{\bst{0.371}} & \scalea{\bst{0.389}}& \scalea{\subbst{0.376}} & \scalea{\subbst{0.391}}& \scalea{0.387} & \scalea{0.396}& \scalea{0.387} & \scalea{0.398}& \scalea{0.411} & \scalea{0.414}& \scalea{0.414} & \scalea{0.421}& \scalea{0.438} & \scalea{0.430}& \scalea{0.396} & \scalea{0.421}& \scalea{0.413} & \scalea{0.407}& \scalea{0.389} & \scalea{0.400}& \scalea{0.403} & \scalea{0.407} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTm2}}
& \scalea{\bst{0.270}} & \scalea{\bst{0.317}}& \scalea{\subbst{0.277}} & \scalea{\subbst{0.321}}& \scalea{0.283} & \scalea{0.331}& \scalea{0.280} & \scalea{0.324}& \scalea{0.295} & \scalea{0.336}& \scalea{0.316} & \scalea{0.365}& \scalea{0.302} & \scalea{0.334}& \scalea{0.308} & \scalea{0.364}& \scalea{0.286} & \scalea{0.328}& \scalea{0.303} & \scalea{0.344}& \scalea{0.342} & \scalea{0.392} \\
\midrule
\multicolumn{2}{l}{\scalea{ETTh1}}
& \scalea{\bst{0.431}} & \scalea{\bst{0.431}}& \scalea{0.449} & \scalea{0.439}& \sc

/tmp/ipykernel_253782/3944324592.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mse_rank'] = dfi.groupby('pred_len')['mse'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfi['mae_rank'] = dfi.groupby('pred_len')['mae'].rank(ascending=True, method='dense')
/tmp/ipykernel_253782/3944324592.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_index